# Prompt 13: Column Manipulation and Expressions
## Databricks Certified Associate Developer for Apache Spark
### Topic 3 — DataFrame and Column Operations (20%)

---

## withColumn — Add or Replace a Column

```python
df.withColumn('new_col', expression)   # adds new column OR replaces existing if same name
```

**Key rules:**
- If `new_col` **doesn't exist** → adds the column
- If `new_col` **already exists** → **replaces** it (no error)
- Returns a **new DataFrame** — original is unchanged
- **Avoid chaining many `withColumn` calls** on wide DataFrames — each adds a projection
  to the plan; prefer a single `select()` for bulk operations

## withColumns (Spark 3.3+)

```python
# Add/replace multiple columns in ONE projection (more efficient than chained withColumn)
df.withColumns({
    'salary_bumped': col('salary') * 1.1,
    'grade':         F.when(col('salary') > 90000, 'senior').otherwise('junior'),
    'name_upper':    F.upper(col('name')),
})
```

## drop — Remove Columns

```python
df.drop('col1', 'col2')            # string names
df.drop(col('col1'), col('col2'))  # Column objects
```

---

## Type Casting

| Method | Syntax | Notes |
|--------|--------|-------|
| `Column.cast(type)` | `col('salary').cast(StringType())` | Uses DataType object |
| `Column.cast(type_str)` | `col('salary').cast('string')` | DDL string (shorter) |
| `Column.astype(type)` | `col('salary').astype('string')` | Alias for `cast()` — identical |
| `selectExpr CAST` | `df.selectExpr('CAST(salary AS STRING)')` | SQL syntax |

**Common type strings:** `'string'`, `'int'`, `'integer'`, `'long'`, `'bigint'`, `'float'`, `'double'`, `'boolean'`, `'date'`, `'timestamp'`, `'decimal(10,2)'`

**Exam tip:** `cast()` and `astype()` are identical — the exam may test both. A failed cast returns `null`, not an error.

In [ ]:
# Cell 1: Setup and withColumn / drop / cast basics
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, BooleanType, LongType, TimestampType
)
import pyspark.sql.functions as F
from pyspark.sql.functions import col, lit, expr, when

spark = SparkSession.builder \
    .appName('ColumnManipulation') \
    .master('local[*]') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

employees = spark.createDataFrame([
    (1,  'Alice',   'Engineering', 95000.0,  True,  '2018-03-15'),
    (2,  'Bob',     'Marketing',   72000.0,  True,  '2020-07-01'),
    (3,  'Carol',   'Engineering', 110000.0, False, '2015-11-20'),
    (4,  'Dave',    'HR',           65000.0, True,  '2022-01-10'),
    (5,  'Eve',     'Engineering', 105000.0, True,  '2017-06-05'),
    (6,  'Frank',   'Marketing',   None,     True,  '2021-09-30'),
], ['id', 'name', 'department', 'salary', 'active', 'hire_date_str'])

print('=== Original DataFrame ===')
employees.printSchema()
employees.show()

# --- withColumn: add a new column ---
print('=== withColumn: add salary_bumped ===')
df1 = employees.withColumn('salary_bumped', col('salary') * 1.1)
df1.select('name', 'salary', 'salary_bumped').show()

# --- withColumn: replace an existing column (same name) ---
print('=== withColumn: replace salary (round to 2 dp) ===')
df2 = employees.withColumn('salary', F.round(col('salary'), 2))
df2.select('name', 'salary').show()

# --- drop: remove columns ---
print('=== drop: remove id and hire_date_str ===')
df3 = employees.drop('id', 'hire_date_str')
df3.printSchema()
df3.show(3)

In [ ]:
# Cell 2: Type casting — cast(), astype(), selectExpr CAST

print('=== cast() with type string ===')
df_cast = employees.withColumn('salary_str',     col('salary').cast('string')) \
                   .withColumn('id_long',         col('id').cast('long')) \
                   .withColumn('active_int',      col('active').cast('int')) \
                   .withColumn('hire_date',       col('hire_date_str').cast('date'))

df_cast.select('name', 'salary', 'salary_str', 'id', 'id_long',
               'active', 'active_int', 'hire_date_str', 'hire_date').show()
df_cast.printSchema()

# astype() is identical to cast()
print('=== astype() — identical to cast() ===')
df_astype = employees.withColumn('salary_str', col('salary').astype('string'))
df_astype.select('name', 'salary', 'salary_str').show(3)

# Failed cast returns null (no error)
print('=== Failed cast returns null ===')
df_bad_cast = spark.createDataFrame([
    ('123',),
    ('abc',),   # 'abc' cannot be cast to int → null
    ('456',),
], ['value_str'])
df_bad_cast.withColumn('value_int', col('value_str').cast('int')).show()

# selectExpr CAST
print('=== selectExpr CAST ===')
employees.selectExpr(
    'name',
    'CAST(salary AS STRING) AS salary_str',
    'CAST(hire_date_str AS DATE) AS hire_date',
    'CAST(active AS INT) AS active_int'
).printSchema()

## lit() — Literal Values as Columns

`lit(value)` wraps a Python literal into a Spark `Column` object so it can be used
anywhere a Column is expected.

```python
from pyspark.sql.functions import lit

df.withColumn('source',       lit('CRM_SYSTEM'))   # string literal
df.withColumn('version',      lit(2))               # int literal
df.withColumn('tax_rate',     lit(0.07))            # float literal
df.withColumn('processed_at', lit(None).cast('timestamp'))  # NULL column
df.withColumn('flag',         lit(True))            # boolean literal
```

**When you must use lit():**
```python
# WRONG — comparing Column to Python int raises TypeError
df.filter(col('status') == 1)          # this actually works (Spark auto-wraps scalars in filter)

# CORRECT — explicit lit() required in withColumn / arithmetic
df.withColumn('discounted', col('price') - lit(10))   # not strictly required here
df.withColumn('total', col('qty') * lit(1.07))        # but good practice to be explicit

# REQUIRED — when passing to functions that expect Column, not Python scalar
df.withColumn('tag', lit('GOLD'))  # ← must be lit()
```

## expr() — SQL Expressions in Column Context

`expr(str)` parses a SQL expression string and returns a `Column` object.

```python
from pyspark.sql.functions import expr

# These are all equivalent:
col('salary') * 1.1
expr('salary * 1.1')

# expr() allows complex SQL expressions inline:
expr('CASE WHEN salary > 90000 THEN "senior" ELSE "junior" END')
expr('to_date(hire_date_str, "yyyy-MM-dd")')
expr('datediff(current_date(), hire_date)')
expr('array_contains(skills, "Python")')
```

In [ ]:
# Cell 3: lit() and expr()

print('=== lit() — add literal constant columns ===')
df_lit = employees.withColumn('data_source',  lit('HR_SYSTEM')) \
                  .withColumn('version',       lit(2)) \
                  .withColumn('tax_rate',      lit(0.07)) \
                  .withColumn('processed',     lit(False)) \
                  .withColumn('null_col',      lit(None).cast('string'))

df_lit.select('name', 'data_source', 'version', 'tax_rate', 'processed', 'null_col').show(3)
df_lit.select('name', 'data_source', 'version', 'tax_rate', 'processed', 'null_col').printSchema()

# lit() with arithmetic
print('=== lit() in arithmetic ===')
employees.withColumn('salary_after_tax',
    col('salary') * (lit(1) - lit(0.30))   # 30% tax
).select('name', 'salary', 'salary_after_tax').show()

print('=== expr() — SQL string expressions as Column ===')
df_expr = employees.withColumn('grade',
                      expr('CASE WHEN salary > 90000 THEN "senior" ELSE "junior" END')) \
                   .withColumn('salary_bumped', expr('salary * 1.1')) \
                   .withColumn('hire_date',     expr('TO_DATE(hire_date_str, "yyyy-MM-dd")'))

df_expr.select('name', 'salary', 'grade', 'salary_bumped', 'hire_date').show()
df_expr.select('name', 'salary', 'grade', 'salary_bumped', 'hire_date').printSchema()

# expr() equivalence with col()
print('=== expr() vs col() — same physical plan ===')
a = employees.withColumn('result', col('salary') * 1.1)
b = employees.withColumn('result', expr('salary * 1.1'))
# Both produce identical results
a.select('name','result').show(3)
b.select('name','result').show(3)

## when() / otherwise() — Conditional Column Expressions

Equivalent to SQL `CASE WHEN ... THEN ... ELSE ... END`.

```python
from pyspark.sql.functions import when

# Basic form
when(condition, value)

# With else
when(condition, value).otherwise(else_value)

# Chained conditions (like CASE WHEN ... WHEN ... ELSE ...)
when(cond1, val1) \
    .when(cond2, val2) \
    .when(cond3, val3) \
    .otherwise(default_val)
```

**Important: without `.otherwise()`, unmatched rows return `null`.**

```python
# Missing otherwise → null for unmatched
when(col('salary') > 100000, 'high')   # rows with salary <= 100000 get null

# With otherwise
when(col('salary') > 100000, 'high').otherwise('standard')   # no nulls
```

### Conditions in when()

```python
when(col('status').isNull(), 'unknown')          # null check
when(col('status').isNotNull(), col('status'))   # pass through non-null value
when(col('dept').isin(['Eng', 'IT']), 'tech')    # isin list
when(col('name').startswith('A'), 'starts_A')    # string predicate
when((col('a') > 1) & (col('b') < 5), 'range')  # compound condition
```

In [ ]:
# Cell 4: when() / otherwise() — conditional expressions

print('=== Basic when().otherwise() ===')
df_cond = employees.withColumn(
    'grade',
    when(col('salary') > 100000, 'senior')
    .when(col('salary') > 80000,  'mid')
    .when(col('salary').isNotNull(), 'junior')
    .otherwise('unknown')        # catches nulls and anything else
)
df_cond.select('name', 'salary', 'grade').show()

print('=== without otherwise — unmatched rows become null ===')
df_no_else = employees.withColumn(
    'is_senior',
    when(col('salary') > 100000, lit(True))
    # no otherwise → False and null salary both become null
)
df_no_else.select('name', 'salary', 'is_senior').show()

print('=== Nested / complex conditions ===')
df_complex = employees.withColumn(
    'category',
    when(
        (col('department') == 'Engineering') & (col('salary') > 100000),
        'senior_engineer'
    ).when(
        col('department') == 'Engineering',
        'engineer'
    ).when(
        col('active') == False,
        'inactive'
    ).otherwise('other')
)
df_complex.select('name', 'department', 'salary', 'active', 'category').show()

print('=== when() returning column value (not just literal) ===')
df_col_val = employees.withColumn(
    'display_name',
    when(col('active'), col('name'))          # active → use real name
    .otherwise(F.concat(col('name'), lit(' [inactive]')))  # inactive → append tag
)
df_col_val.select('name', 'active', 'display_name').show()

print('=== when() with isin and isNull ===')
employees.withColumn(
    'dept_group',
    when(col('department').isin('Engineering', 'HR'), 'internal')
    .when(col('salary').isNull(), 'no_data')
    .otherwise('external')
).select('name', 'department', 'salary', 'dept_group').show()

## Arithmetic and String Transformations

### Arithmetic on Columns

| Operation | Syntax | Notes |
|-----------|--------|-------|
| Addition | `col('a') + col('b')` or `col('a') + 10` | |
| Subtraction | `col('a') - col('b')` | |
| Multiplication | `col('a') * 1.1` | |
| Division | `col('a') / col('b')` | Returns `DoubleType`; divide-by-zero → `null` |
| Integer division | `col('a').bitwiseAND(col('b'))` | Use `F.floor(col('a') / col('b'))` |
| Modulo | `col('a') % col('b')` | |
| Power | `col('a') ** 2` or `F.pow(col('a'), 2)` | |
| Absolute value | `F.abs(col('a'))` | |
| Round | `F.round(col('a'), 2)` | Banker's rounding |
| Floor / Ceil | `F.floor(col('a'))`, `F.ceil(col('a'))` | |
| Sqrt | `F.sqrt(col('a'))` | |

### Common String Functions

| Function | Example |
|----------|---------|
| `upper` / `lower` | `F.upper(col('name'))` |
| `trim` / `ltrim` / `rtrim` | `F.trim(col('str'))` |
| `length` | `F.length(col('str'))` |
| `substring` | `F.substring(col('str'), pos, len)` — 1-indexed |
| `concat` | `F.concat(col('a'), lit('-'), col('b'))` |
| `concat_ws` | `F.concat_ws('-', col('a'), col('b'))` — separator first |
| `split` | `F.split(col('str'), pattern)` → ArrayType |
| `regexp_replace` | `F.regexp_replace(col('str'), pattern, replacement)` |
| `regexp_extract` | `F.regexp_extract(col('str'), pattern, group_idx)` |
| `like` / `rlike` | `col('str').like('A%')`, `col('str').rlike('^A.*')` |
| `startswith` / `endswith` | `col('str').startswith('A')` |
| `contains` | `col('str').contains('xyz')` |
| `lpad` / `rpad` | `F.lpad(col('str'), 10, '0')` |
| `replace` | `F.regexp_replace(col('str'), 'old', 'new')` |

In [ ]:
# Cell 5: Arithmetic column expressions

print('=== Arithmetic operations ===')
df_arith = employees.withColumn('salary_10pct_raise', col('salary') * 1.1) \
                    .withColumn('salary_bonus',        col('salary') + lit(5000)) \
                    .withColumn('annual_after_tax',    F.round(col('salary') * lit(0.7), 2)) \
                    .withColumn('monthly_salary',      F.round(col('salary') / lit(12), 2)) \
                    .withColumn('salary_squared',      F.pow(col('salary'), 2)) \
                    .withColumn('salary_sqrt',         F.round(F.sqrt(col('salary')), 4))

df_arith.select(
    'name', 'salary', 'salary_10pct_raise', 'salary_bonus',
    'annual_after_tax', 'monthly_salary', 'salary_sqrt'
).show()

# Divide by zero → null (no error)
print('=== Divide by zero returns null ===')
spark.createDataFrame([(10.0, 2.0), (5.0, 0.0), (0.0, 3.0)], ['a', 'b']) \
     .withColumn('result', col('a') / col('b')) \
     .show()

# Arithmetic with null propagates null
print('=== null propagation in arithmetic ===')
employees.withColumn('salary_bumped', col('salary') * 1.1) \
         .select('name', 'salary', 'salary_bumped') \
         .show()  # Frank has null salary → null salary_bumped

In [ ]:
# Cell 6: String transformation functions

df_str = spark.createDataFrame([
    (1, '  Alice Smith  ', 'alice.smith@example.com', '2024-03-15'),
    (2, 'Bob Jones',       'bob.jones@test.org',      '2023-11-01'),
    (3, 'CAROL BROWN',     'carol@EXAMPLE.COM',       '2022-07-20'),
    (4, 'dave',            None,                      '2021-04-08'),
], ['id', 'full_name', 'email', 'date_str'])

print('=== String transformation functions ===')
df_transformed = df_str \
    .withColumn('name_trimmed',   F.trim(col('full_name'))) \
    .withColumn('name_upper',     F.upper(F.trim(col('full_name')))) \
    .withColumn('name_lower',     F.lower(F.trim(col('full_name')))) \
    .withColumn('name_len',       F.length(F.trim(col('full_name')))) \
    .withColumn('first_name',     F.split(F.trim(col('full_name')), ' ')[0]) \
    .withColumn('email_lower',    F.lower(col('email'))) \
    .withColumn('email_domain',   F.regexp_extract(col('email'), r'@(.+)$', 1)) \
    .withColumn('email_masked',   F.regexp_replace(col('email'), r'(?<=.).(?=.*@)', '*')) \
    .withColumn('name_padded',    F.rpad(F.trim(col('full_name')), 20, '.')) \
    .withColumn('initials',       F.substring(F.trim(col('full_name')), 1, 1))

df_transformed.select(
    'full_name', 'name_trimmed', 'name_upper', 'name_lower', 'name_len',
    'first_name', 'email_lower', 'email_domain', 'name_padded', 'initials'
).show(truncate=False)

# concat and concat_ws
print('=== concat and concat_ws ===')
df_str \
    .withColumn('greeting',     F.concat(lit('Hello, '), F.trim(col('full_name')), lit('!'))) \
    .withColumn('id_name',      F.concat_ws('_', col('id').cast('string'), F.trim(col('full_name')))) \
    .select('full_name', 'greeting', 'id_name') \
    .show(truncate=False)

# like / rlike
print('=== like / rlike / contains / startswith ===')
df_str \
    .withColumn('starts_with_A',   col('full_name').startswith('A')) \
    .withColumn('ends_with_es',    col('full_name').endswith('es')) \
    .withColumn('contains_smith',  col('full_name').contains('Smith')) \
    .withColumn('like_pattern',    col('full_name').like('%Smith%')) \
    .withColumn('rlike_pattern',   col('full_name').rlike('^[A-Z].*')) \
    .select('full_name', 'starts_with_A', 'ends_with_es', 'contains_smith', 'rlike_pattern') \
    .show(truncate=False)

In [ ]:
# Cell 7: Combining everything — complex multi-step transformation pipeline

# Simulate a raw ingest scenario: messy data needing standardization
raw = spark.createDataFrame([
    (1, '  Alice  ', 'engineering', '95000', '1', '2018-03-15'),
    (2, 'BOB',       'Marketing',   '72000', '1', '2020-07-01'),
    (3, 'carol',     'ENGINEERING', '110000','0', '2015-11-20'),
    (4, 'Dave',      'hr',          'n/a',   '1', '2022-01-10'),
    (5, 'Eve ',      'Engineering', '105000','1', 'bad_date'),
], ['id', 'raw_name', 'raw_dept', 'raw_salary', 'raw_active', 'raw_hire'])

print('=== Raw messy data ===')
raw.show()

# --- Standardization pipeline ---
clean = raw \
    .withColumn('name',         F.initcap(F.trim(col('raw_name')))) \
    .withColumn('department',   F.initcap(F.lower(F.trim(col('raw_dept'))))) \
    .withColumn('salary',       when(
                                    col('raw_salary').rlike(r'^\d+$'),
                                    col('raw_salary').cast('double')
                                ).otherwise(lit(None).cast('double'))) \
    .withColumn('active',       col('raw_active').cast('boolean')) \
    .withColumn('hire_date',    col('raw_hire').cast('date')) \
    .withColumn('grade',        when(col('salary') > 100000, 'senior')
                                .when(col('salary') > 80000,  'mid')
                                .when(col('salary').isNotNull(), 'junior')
                                .otherwise('unknown')) \
    .withColumn('salary_monthly', F.round(col('salary') / lit(12), 2)) \
    .withColumn('data_source',  lit('HR_IMPORT')) \
    .drop('raw_name', 'raw_dept', 'raw_salary', 'raw_active', 'raw_hire')

print('=== Cleaned and standardized data ===')
clean.printSchema()
clean.show(truncate=False)

spark.stop()
print('Done.')

## Quick Reference Summary

### Core Column Operations

```python
# Add/replace column
df.withColumn('new_col', expression)                           # single column
df.withColumns({'c1': expr1, 'c2': expr2})                    # multiple (Spark 3.3+)

# Remove column
df.drop('col1', 'col2')

# Cast type
col('salary').cast('string')             # or .cast(StringType())
col('salary').astype('string')           # identical to cast()
# failed cast → null (no error)

# Literal value
lit('GOLD'), lit(42), lit(None).cast('string')

# SQL expression as Column
expr('salary * 1.1')                     # equivalent to col('salary') * 1.1
expr('CASE WHEN salary > 90000 THEN "senior" ELSE "junior" END')

# Conditional
when(condition, val1).when(cond2, val2).otherwise(default)
# no otherwise → unmatched rows return null
```

### Arithmetic
```python
col('a') + col('b'), col('a') - col('b'), col('a') * 1.1, col('a') / col('b')
col('a') % col('b')     # modulo
F.pow(col('a'), 2)      # power
F.abs(col('a'))         # absolute
F.round(col('a'), 2)    # round
F.floor(col('a'))       # floor
F.ceil(col('a'))        # ceiling
F.sqrt(col('a'))        # square root
# null in arithmetic → null result (null propagates)
# divide by zero → null (no error)
```

### Key String Functions
```python
F.upper(col), F.lower(col), F.initcap(col)
F.trim(col), F.ltrim(col), F.rtrim(col)
F.length(col)
F.substring(col, 1, 3)           # 1-indexed; extracts 3 chars from position 1
F.concat(col1, lit('-'), col2)
F.concat_ws('-', col1, col2)     # separator is FIRST argument
F.split(col, pattern)            # returns ArrayType
F.regexp_replace(col, pattern, replacement)
F.regexp_extract(col, pattern, group_index)
col.like('A%'), col.rlike('^A.*'), col.contains('x'), col.startswith('A'), col.endswith('z')
F.lpad(col, length, pad), F.rpad(col, length, pad)
```

## Real-World Scenarios

<details>
<summary>Scenario 1: Standardising a raw ingested DataFrame (null-safe salary parsing)</summary>

**Situation:**
A pipeline ingests CSV files where `salary` is a string column containing either a numeric
value like `'95000'` or invalid strings like `'n/a'`, `'TBD'`, or empty strings.
Direct `cast('double')` would silently produce nulls for valid numbers too (via failed cast).
The engineer needs a null-safe pattern.

**Code:**
```python
from pyspark.sql.functions import when, col, lit

# Validate before casting using rlike — only cast if the string is purely numeric
df.withColumn(
    'salary',
    when(
        col('raw_salary').rlike(r'^\d+(\.\d+)?$'),  # matches '95000' or '95000.5'
        col('raw_salary').cast('double')
    ).otherwise(lit(None).cast('double'))            # invalid → explicit null
)

# Alternative: just cast and accept that invalid strings become null
df.withColumn('salary', col('raw_salary').cast('double'))  # 'n/a' → null automatically
```

**Expected Outcome:** Clean `DoubleType` salary column where invalid entries are `null` with clear intent.
**Exam Sub-topic:** `cast()`, `when()`, `rlike`, null handling
</details>

<details>
<summary>Scenario 2: Adding audit columns to every DataFrame in a pipeline</summary>

**Situation:**
Every DataFrame produced by a pipeline must include three audit columns:
`loaded_at` (current timestamp), `data_source` (string), and `pipeline_version` (int).

**Code:**
```python
from pyspark.sql.functions import current_timestamp, lit

def add_audit_columns(df, source_name: str, version: int):
    return df \
        .withColumn('loaded_at',        current_timestamp()) \
        .withColumn('data_source',      lit(source_name)) \
        .withColumn('pipeline_version', lit(version))

# Apply to any DataFrame
enriched = add_audit_columns(customers_df, 'CRM_EXPORT', 3)
enriched.printSchema()
# ... + loaded_at TIMESTAMP, data_source STRING, pipeline_version INT
```

**Expected Outcome:** Reusable function using `lit()` and `current_timestamp()` adds three audit columns.
**Exam Sub-topic:** `lit()`, `withColumn()`, `current_timestamp()`
</details>

<details>
<summary>Scenario 3: Salary band classification with chained when().when().otherwise()</summary>

**Situation:**
An HR analytics team needs to segment employees into pay bands: `'Band 1'` (< 50k),
`'Band 2'` (50k–80k), `'Band 3'` (80k–100k), `'Band 4'` (> 100k), `'Unclassified'` (null salary).

**Code:**
```python
from pyspark.sql.functions import when, col

df.withColumn(
    'pay_band',
    when(col('salary').isNull(),            'Unclassified')
    .when(col('salary') < 50000,            'Band 1')
    .when((col('salary') >= 50000) & (col('salary') < 80000),  'Band 2')
    .when((col('salary') >= 80000) & (col('salary') < 100000), 'Band 3')
    .otherwise('Band 4')
)
```

**Expected Outcome:** Clean categorical column `pay_band` with no nulls.
**Exam Sub-topic:** `when()`, chained conditions, `isNull()`, `&` operator
</details>

<details>
<summary>Scenario 4: Extracting structured fields from a free-text email column</summary>

**Situation:**
A DataFrame has an `email` column. The pipeline needs to extract the username and domain
as separate columns using regex, then mask the local part of the email for GDPR compliance.

**Code:**
```python
from pyspark.sql.functions import regexp_extract, regexp_replace, lower, col

df.withColumn('email_lc',    lower(col('email'))) \
  .withColumn('username',    regexp_extract(col('email_lc'), r'^([^@]+)@', 1)) \
  .withColumn('domain',      regexp_extract(col('email_lc'), r'@(.+)$', 1)) \
  .withColumn('email_masked',
              regexp_replace(col('email_lc'), r'(?<=.)(?=.*@)', '*'))
# a.smith@example.com → a*****@example.com
```

**Expected Outcome:** Three new columns: normalised email, extracted username, extracted domain, and GDPR-safe masked email.
**Exam Sub-topic:** `regexp_extract`, `regexp_replace`, `lower`, `withColumn`
</details>

<details>
<summary>Scenario 5: Using expr() to apply date arithmetic without changing the API style</summary>

**Situation:**
A pipeline needs to add `days_employed` (days since hire date), `hire_year`, and
a `review_due_date` (hire date + 365 days) to an employee DataFrame.
The team prefers SQL-style expressions over DataFrame function calls for readability.

**Code:**
```python
from pyspark.sql.functions import expr, col

df.withColumn('hire_date',       col('hire_date_str').cast('date')) \
  .withColumn('days_employed',   expr('datediff(current_date(), hire_date)')) \
  .withColumn('hire_year',       expr('year(hire_date)')) \
  .withColumn('review_due_date', expr('date_add(hire_date, 365)'))

# Equivalent using F.* functions:
# .withColumn('days_employed',   F.datediff(F.current_date(), col('hire_date')))
# .withColumn('hire_year',       F.year(col('hire_date')))
# .withColumn('review_due_date', F.date_add(col('hire_date'), 365))
```

**Expected Outcome:** Three new date-derived columns. `expr()` enables SQL syntax within DataFrame API — same physical plan as `F.*` equivalent.
**Exam Sub-topic:** `expr()`, date functions, `cast('date')`
</details>

## Exam Cheat Sheet

| Concept | Key Exam Fact |
|---------|---------------|
| `withColumn('existing', expr)` | **Replaces** the column — no error, no duplicate |
| `withColumn('new', expr)` | Adds a new column; all other columns kept |
| `withColumns({...})` | Spark 3.3+: adds/replaces multiple columns in one projection |
| `df.drop('c1','c2')` | Removes columns; safe if column doesn't exist (no error in most versions) |
| `cast()` vs `astype()` | Identical — `astype` is an alias for `cast` |
| Failed cast | Returns `null`, **not** an exception |
| `lit(value)` | Wraps Python scalar into a `Column`; needed for `None` → `lit(None).cast(type)` |
| `expr('sql string')` | Parses SQL string → `Column`; equivalent to `col()` + operators |
| `when(cond, val)` | Returns `null` for unmatched rows if no `.otherwise()` |
| `.otherwise(val)` | Provides the ELSE value; prevents unmatched rows from being `null` |
| Null in arithmetic | `null + anything = null`; propagates through all operators |
| Division by zero | Returns `null` — no `ZeroDivisionError` |
| `F.concat_ws(sep, c1, c2)` | Separator is the **first** argument; skips nulls |
| `F.split(col, pattern)` | Returns `ArrayType`; use `[0]` to get first element |
| `F.substring(col, pos, len)` | **1-indexed** position (unlike Python's 0-indexed) |
| `col.like('A%')` | SQL LIKE pattern (`%` = wildcard) |
| `col.rlike('^A.*')` | Java regex pattern |
| `F.initcap(col)` | Capitalises first letter of each word |
| DataFrames are **immutable** | All transformations return a new DataFrame |

---

## Practice Q&A

<details>
<summary>Q1: What happens when you call withColumn() using a column name that already exists in the DataFrame?</summary>

**A:** The existing column is **replaced** with the new expression. No error is raised, no duplicate column is created. The operation returns a new DataFrame where the named column now contains the result of the expression. This is a common source of bugs — always check you're not accidentally overwriting a column you still need.
</details>

<details>
<summary>Q2: What is the difference between cast() and astype()?</summary>

**A:** They are **identical** — `astype()` is an alias for `cast()`. Both accept a `DataType` object (e.g., `StringType()`) or a DDL type string (e.g., `'string'`). Both return `null` (not an error) when the conversion fails for a given row.
</details>

<details>
<summary>Q3: When is lit() required?</summary>

**A:** `lit(value)` is required wherever Spark expects a `Column` object but you want to provide a constant Python value. Key cases:
- `withColumn('tag', lit('GOLD'))` — adding a constant string column
- `lit(None).cast('string')` — adding a `null` column with a specific type
- Passing to functions that accept only `Column` objects

In simple arithmetic expressions like `col('salary') * 1.1`, Spark auto-wraps the scalar `1.1` — `lit()` is not strictly required there but using it is explicit and correct.
</details>

<details>
<summary>Q4: What is returned for rows that don't match any condition in when() if there is no otherwise()?</summary>

**A:** `null`. If no `when()` condition matches and there is no `.otherwise()`, Spark returns `null` for that row. Always add `.otherwise(default_value)` when you want every row to have a non-null result. This is a very common exam question.
</details>

<details>
<summary>Q5: What is the difference between F.concat() and F.concat_ws()?</summary>

**A:**
- `F.concat(col1, col2, ...)` — concatenates columns directly; returns `null` if **any** input is `null`
- `F.concat_ws(separator, col1, col2, ...)` — concatenates with separator **and skips null values**; separator is the **first** argument

```python
# concat: null propagates
F.concat(lit('Hello'), lit(None), lit('World'))  # → null

# concat_ws: nulls skipped
F.concat_ws(' ', lit('Hello'), lit(None), lit('World'))  # → 'Hello World'
```

For building full names or addresses where some parts may be null, prefer `concat_ws`.
</details>